# Day 6：手写 MoE 与分布式训练实践（本周核心实践 II！）

> **目标**：从零实现 MoE 架构的核心组件——Router（Top-K Gating）、Expert（FFN）、MoE Layer（含负载均衡 Loss）、完整 MoE Transformer Block；验证路由分布和负载均衡效果；模拟 Expert Parallelism 的 AllToAll 通信；完成端到端 MoE 训练循环。

**实现路线图**：

```
Part 1: 环境与 MoE 配置
  ↓
Part 2: 手写 Router（Top-K Gating）
  ↓
Part 3: 手写 Expert（标准 FFN Expert）
  ↓
Part 4: 手写 MoE Layer（Router + Experts + 负载均衡 Loss）
  ↓
Part 5: 手写完整 MoE Transformer Block
  ↓
Part 6: 验证 — Routing 分布可视化
  ↓
Part 7: 验证 — 负载均衡 Loss 有效性
  ↓
Part 8: MoE 参数量分析（总参数 vs 激活参数）
  ↓
Part 9: 模拟 Expert Parallelism（AllToAll）
  ↓
Part 10: 端到端 MoE 训练循环
```

## Part 1：环境与 MoE 配置

定义 MoE 架构的超参数配置。关键参数：
- `num_experts`：专家数量 $E$
- `top_k`：每个 token 激活的专家数 $K$
- `aux_loss_alpha`：辅助负载均衡损失系数 $\alpha$

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
import numpy as np
from dataclasses import dataclass
from typing import List, Optional, Tuple, Dict

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


@dataclass
class MoEConfig:
    """MoE 模型配置"""
    d_model: int = 256          # 隐层维度
    n_heads: int = 8            # 注意力头数
    n_layers: int = 4           # Transformer 层数
    num_experts: int = 8        # 专家数量 E
    num_shared_experts: int = 0 # 共享专家数（0 = 无共享专家）
    top_k: int = 2              # 每 token 激活 K 个专家
    expert_ffn_dim: int = 512   # 每个专家的 FFN 中间维度
    aux_loss_alpha: float = 0.01  # 辅助 loss 系数
    z_loss_weight: float = 0.001  # Z-Loss 系数
    vocab_size: int = 1000      # 词表大小
    max_seq_len: int = 128      # 最大序列长度
    dropout: float = 0.1        # Dropout
    moe_every_n_layers: int = 2 # 每隔 n 层放一层 MoE（交替）


config = MoEConfig()
print("MoE 配置:")
for k, v in config.__dict__.items():
    print(f"  {k}: {v}")
print(f"\n每 token 激活 {config.top_k}/{config.num_experts} 个专家")
print(f"激活比例: {config.top_k / config.num_experts:.1%}")

## Part 2：手写 Router（Top-K Gating）

Router 是 MoE 的核心组件，决定每个 token 分配给哪些专家。

**Top-K 路由数学**：

$$\text{logits} = x \cdot W_g \in \mathbb{R}^{E}$$

$$\text{gates} = \text{Softmax}(\text{logits})$$

$$\text{TopK\_indices}, \text{TopK\_weights} = \text{TopK}(\text{gates}, k=K)$$

Top-K 后重新归一化权重：$w_i' = \frac{w_i}{\sum_{j \in \text{TopK}} w_j}$

In [ ]:
class TopKRouter(nn.Module):
    """
    Top-K 路由器：每个 token 选择 Top-K 个专家。
    
    包含：
    - Top-K 路由（Softmax → TopK → 归一化）
    - Noisy gating（训练时添加噪声促进探索）
    - 辅助负载均衡 Loss
    - Z-Loss（惩罚过大 logits）
    """
    def __init__(self, d_model: int, num_experts: int, top_k: int,
                 aux_loss_alpha: float = 0.01, z_loss_weight: float = 0.001):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.aux_loss_alpha = aux_loss_alpha
        self.z_loss_weight = z_loss_weight
        
        self.gate = nn.Linear(d_model, num_experts, bias=False)
        self.noise_gate = nn.Linear(d_model, num_experts, bias=False)
    
    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, Dict[str, torch.Tensor]]:
        """
        Args:
            x: [batch * seq_len, d_model] — 扁平化的 token 表示
        Returns:
            topk_indices: [batch * seq_len, top_k] — 选中的专家索引
            topk_weights: [batch * seq_len, top_k] — 归一化的门控权重
            aux_losses: dict — 辅助损失
        """
        # 路由 logits
        logits = self.gate(x)  # [T, E]
        
        # 训练时添加噪声（Noisy Top-K Gating）
        if self.training:
            noise_std = F.softplus(self.noise_gate(x))  # [T, E]
            noise = torch.randn_like(logits) * noise_std
            logits = logits + noise
        
        # Softmax 得到门控概率
        gates = F.softmax(logits, dim=-1)  # [T, E]
        
        # Top-K 选择
        topk_weights, topk_indices = torch.topk(gates, self.top_k, dim=-1)  # [T, K]
        
        # 归一化权重（使 Top-K 权重之和为 1）
        topk_weights = topk_weights / (topk_weights.sum(dim=-1, keepdim=True) + 1e-9)
        
        # 计算辅助损失
        aux_losses = self._compute_aux_losses(gates, topk_indices, logits)
        
        return topk_indices, topk_weights, aux_losses
    
    def _compute_aux_losses(self, gates: torch.Tensor, topk_indices: torch.Tensor,
                            logits: torch.Tensor) -> Dict[str, torch.Tensor]:
        """计算辅助负载均衡 Loss 和 Z-Loss"""
        T = gates.shape[0]
        losses = {}
        
        # 1. 辅助负载均衡 Loss
        # f_i: 每个专家实际被选中的 token 比例（用 Top-1 统计）
        top1_indices = topk_indices[:, 0]  # [T]
        one_hot = F.one_hot(top1_indices, self.num_experts).float()  # [T, E]
        f = one_hot.mean(dim=0)  # [E] — 每个专家被选为 Top-1 的比例
        
        # P_i: 路由器给每个专家的平均概率
        P = gates.mean(dim=0)  # [E]
        
        # L_aux = alpha * E * sum(f_i * P_i)
        load_balance_loss = self.aux_loss_alpha * self.num_experts * (f * P).sum()
        losses['load_balance'] = load_balance_loss
        
        # 2. Z-Loss: 惩罚 logits 过大
        # L_z = mean(log(sum(exp(z_i)))^2)
        log_sum_exp = torch.logsumexp(logits, dim=-1)  # [T]
        z_loss = self.z_loss_weight * (log_sum_exp ** 2).mean()
        losses['z_loss'] = z_loss
        
        # 记录专家利用率统计
        losses['expert_counts'] = one_hot.sum(dim=0)  # [E]
        
        return losses


# 验证 Router
print("=" * 50)
print("Top-K Router 验证")
print("=" * 50)

router = TopKRouter(config.d_model, config.num_experts, config.top_k,
                    config.aux_loss_alpha, config.z_loss_weight)

x_test = torch.randn(32, config.d_model)  # 32 个 token
router.train()
topk_idx, topk_w, aux = router(x_test)

print(f"\n输入 shape: {x_test.shape}")
print(f"TopK indices shape: {topk_idx.shape}  (期望 [32, {config.top_k}])")
print(f"TopK weights shape: {topk_w.shape}")
print(f"\n前 5 个 token 的路由:")
for i in range(5):
    experts = topk_idx[i].tolist()
    weights = topk_w[i].tolist()
    print(f"  Token {i}: Expert {experts}, Weights {[f'{w:.3f}' for w in weights]}")

print(f"\n辅助损失:")
print(f"  Load Balance Loss: {aux['load_balance'].item():.6f}")
print(f"  Z-Loss: {aux['z_loss'].item():.6f}")
print(f"\n各专家被选为 Top-1 的次数: {aux['expert_counts'].int().tolist()}")

# 检查权重归一化
weight_sums = topk_w.sum(dim=-1)
assert torch.allclose(weight_sums, torch.ones_like(weight_sums), atol=1e-5), "权重未归一化!"
print(f"\n✅ 权重归一化验证通过 (每行权重和 ≈ 1.0)")

## Part 3：手写 Expert（标准 FFN Expert）

每个 Expert 就是一个标准的 FFN，结构与 Transformer FFN 一致。

我们实现两种：
1. **标准 FFN**: $\text{Expert}(x) = \text{GELU}(xW_1 + b_1)W_2 + b_2$
2. **SwiGLU FFN**: $\text{Expert}(x) = (\text{SiLU}(xW_{\text{gate}}) \odot xW_{\text{up}}) \cdot W_{\text{down}}$

In [ ]:
class Expert(nn.Module):
    """标准 FFN Expert (GELU 激活)"""
    def __init__(self, d_model: int, ffn_dim: int, dropout: float = 0.1):
        super().__init__()
        self.w1 = nn.Linear(d_model, ffn_dim)
        self.w2 = nn.Linear(ffn_dim, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w2(self.dropout(F.gelu(self.w1(x))))


class SwiGLUExpert(nn.Module):
    """SwiGLU FFN Expert (LLaMA / Mixtral 风格)"""
    def __init__(self, d_model: int, ffn_dim: int, dropout: float = 0.1):
        super().__init__()
        self.w_gate = nn.Linear(d_model, ffn_dim, bias=False)
        self.w_up = nn.Linear(d_model, ffn_dim, bias=False)
        self.w_down = nn.Linear(ffn_dim, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w_down(self.dropout(F.silu(self.w_gate(x)) * self.w_up(x)))


# 验证 Expert
print("=" * 50)
print("Expert 验证")
print("=" * 50)

expert_gelu = Expert(config.d_model, config.expert_ffn_dim)
expert_swiglu = SwiGLUExpert(config.d_model, config.expert_ffn_dim)

x_test = torch.randn(4, config.d_model)

out_gelu = expert_gelu(x_test)
out_swiglu = expert_swiglu(x_test)

print(f"\n输入 shape: {x_test.shape}")
print(f"GELU Expert 输出 shape: {out_gelu.shape}  (期望 [4, {config.d_model}])")
print(f"SwiGLU Expert 输出 shape: {out_swiglu.shape}")

# 参数量对比
gelu_params = sum(p.numel() for p in expert_gelu.parameters())
swiglu_params = sum(p.numel() for p in expert_swiglu.parameters())
print(f"\nGELU Expert 参数量: {gelu_params:,}")
print(f"SwiGLU Expert 参数量: {swiglu_params:,}")
print(f"SwiGLU / GELU 参数比: {swiglu_params/gelu_params:.2f}x (SwiGLU 多一个 gate 投影)")

assert out_gelu.shape == (4, config.d_model), "输出维度不匹配!"
print(f"\n✅ Expert 验证通过!")

## Part 4：手写 MoE Layer

MoE Layer 将 Router 和 Experts 组合在一起：

$$y = \sum_{i \in \text{TopK}} g_i(x) \cdot \text{Expert}_i(x)$$

实现的关键挑战是**高效分发**：将 token 路由到对应专家，计算后再组合回来。

简单实现策略（循环遍历专家）：
1. 对每个专家 $i$，找出所有被路由到它的 token
2. 批量计算 Expert_i(tokens)
3. 将结果加权写回输出

In [ ]:
class MoELayer(nn.Module):
    """
    完整的 MoE Layer：Router + 多个 Expert + 可选 Shared Expert。
    
    实现方式：按专家遍历（简单直观），适合理解原理。
    生产级实现（如 Megablocks）会用分组 GEMM 提升效率。
    """
    def __init__(self, config: MoEConfig):
        super().__init__()
        self.config = config
        
        # Router
        self.router = TopKRouter(
            config.d_model, config.num_experts, config.top_k,
            config.aux_loss_alpha, config.z_loss_weight
        )
        
        # 路由专家
        self.experts = nn.ModuleList([
            SwiGLUExpert(config.d_model, config.expert_ffn_dim, config.dropout)
            for _ in range(config.num_experts)
        ])
        
        # 共享专家（可选）
        self.shared_experts = None
        if config.num_shared_experts > 0:
            self.shared_experts = nn.ModuleList([
                SwiGLUExpert(config.d_model, config.expert_ffn_dim, config.dropout)
                for _ in range(config.num_shared_experts)
            ])
    
    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        """
        Args:
            x: [batch, seq_len, d_model]
        Returns:
            output: [batch, seq_len, d_model]
            aux_losses: dict — 辅助损失
        """
        batch_size, seq_len, d_model = x.shape
        
        # 扁平化: [B, S, D] → [B*S, D]
        x_flat = x.view(-1, d_model)  # [T, D] where T = B*S
        T = x_flat.shape[0]
        
        # 路由
        topk_indices, topk_weights, aux_losses = self.router(x_flat)  # [T, K]
        
        # 初始化输出
        output = torch.zeros_like(x_flat)  # [T, D]
        
        # 对每个专家，找出被路由到它的 token 并计算
        for expert_idx in range(self.config.num_experts):
            expert = self.experts[expert_idx]
            
            # 找出哪些 (token, k) 对选择了这个专家
            # topk_indices: [T, K]，需要找出所有 == expert_idx 的位置
            mask = (topk_indices == expert_idx)  # [T, K] bool
            token_mask = mask.any(dim=-1)  # [T] — 哪些 token 使用了这个专家
            
            if not token_mask.any():
                continue
            
            # 提取这些 token
            selected_tokens = x_flat[token_mask]  # [n_selected, D]
            
            # 计算专家输出
            expert_output = expert(selected_tokens)  # [n_selected, D]
            
            # 获取对应权重：对于选中了此专家的 token，找到对应的 top-k 权重
            # mask[token_mask] 是 [n_selected, K]，取其中 True 位置的权重
            selected_mask = mask[token_mask]  # [n_selected, K]
            selected_weights = topk_weights[token_mask]  # [n_selected, K]
            weights = (selected_weights * selected_mask.float()).sum(dim=-1, keepdim=True)  # [n_selected, 1]
            
            # 加权写回
            output[token_mask] += weights * expert_output
        
        # 共享专家（如果有）：每个 token 都经过共享专家
        if self.shared_experts is not None:
            for shared_expert in self.shared_experts:
                output += shared_expert(x_flat)
        
        # 恢复 shape: [T, D] → [B, S, D]
        output = output.view(batch_size, seq_len, d_model)
        
        return output, aux_losses


# 验证 MoE Layer
print("=" * 50)
print("MoE Layer 验证")
print("=" * 50)

moe_layer = MoELayer(config)
x_test = torch.randn(2, 16, config.d_model)  # batch=2, seq=16

moe_layer.train()
output, aux = moe_layer(x_test)

print(f"\n输入 shape: {x_test.shape}")
print(f"输出 shape: {output.shape}  (期望 {x_test.shape})")
print(f"Load Balance Loss: {aux['load_balance'].item():.6f}")
print(f"Z-Loss: {aux['z_loss'].item():.6f}")
print(f"各专家 Top-1 选中次数: {aux['expert_counts'].int().tolist()}")
print(f"总 tokens: {2 * 16} = batch({2}) × seq({16})")

assert output.shape == x_test.shape, "输出维度不匹配!"
print(f"\n✅ MoE Layer 验证通过!")

## Part 5：手写完整 MoE Transformer Block

将 MoE Layer 嵌入到 Transformer Block 中，替换标准 FFN：

```
MoE Transformer Block:
  x → LayerNorm → Attention → + Residual → LayerNorm → MoE Layer → + Residual → y
```

同时实现标准 FFN Block 用于对比（交替放置时的非 MoE 层）。

In [ ]:
class RMSNorm(nn.Module):
    """RMSNorm (LLaMA 风格)"""
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    
    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x / rms * self.weight


class MultiHeadAttention(nn.Module):
    """标准多头注意力（简化版，用于 MoE Block 演示）"""
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.wq = nn.Linear(d_model, d_model, bias=False)
        self.wk = nn.Linear(d_model, d_model, bias=False)
        self.wv = nn.Linear(d_model, d_model, bias=False)
        self.wo = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
    
    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        B, S, D = x.shape
        q = self.wq(x).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.wk(x).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.wv(x).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = self.attn_dropout(F.softmax(scores, dim=-1))
        out = (attn @ v).transpose(1, 2).contiguous().view(B, S, D)
        return self.wo(out)


class MoETransformerBlock(nn.Module):
    """MoE Transformer Block: Attention + MoE Layer"""
    def __init__(self, config: MoEConfig, use_moe: bool = True):
        super().__init__()
        self.use_moe = use_moe
        
        self.attn_norm = RMSNorm(config.d_model)
        self.attention = MultiHeadAttention(config.d_model, config.n_heads, config.dropout)
        
        self.ffn_norm = RMSNorm(config.d_model)
        if use_moe:
            self.ffn = MoELayer(config)
        else:
            self.ffn = SwiGLUExpert(config.d_model, config.expert_ffn_dim, config.dropout)
    
    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None
                ) -> Tuple[torch.Tensor, Optional[Dict[str, torch.Tensor]]]:
        # Attention with residual
        h = x + self.attention(self.attn_norm(x), mask)
        
        # FFN / MoE with residual
        normed = self.ffn_norm(h)
        if self.use_moe:
            ffn_out, aux_losses = self.ffn(normed)
        else:
            ffn_out = self.ffn(normed)
            aux_losses = None
        
        out = h + ffn_out
        return out, aux_losses


class MoETransformer(nn.Module):
    """完整的 MoE Transformer 模型（交替放置 MoE 层）"""
    def __init__(self, config: MoEConfig):
        super().__init__()
        self.config = config
        
        self.embedding = nn.Embedding(config.vocab_size, config.d_model)
        
        self.layers = nn.ModuleList()
        for i in range(config.n_layers):
            use_moe = (i % config.moe_every_n_layers == config.moe_every_n_layers - 1)
            self.layers.append(MoETransformerBlock(config, use_moe=use_moe))
        
        self.norm = RMSNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
    
    def forward(self, input_ids: torch.Tensor, mask: Optional[torch.Tensor] = None
                ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        x = self.embedding(input_ids)  # [B, S, D]
        
        all_aux_losses = []
        for layer in self.layers:
            x, aux = layer(x, mask)
            if aux is not None:
                all_aux_losses.append(aux)
        
        x = self.norm(x)
        logits = self.lm_head(x)  # [B, S, vocab]
        
        # 合并所有 MoE 层的辅助损失
        combined_aux = {}
        if all_aux_losses:
            combined_aux['load_balance'] = sum(a['load_balance'] for a in all_aux_losses) / len(all_aux_losses)
            combined_aux['z_loss'] = sum(a['z_loss'] for a in all_aux_losses) / len(all_aux_losses)
            combined_aux['expert_counts'] = torch.stack([a['expert_counts'] for a in all_aux_losses])
        
        return logits, combined_aux


# 验证完整模型
print("=" * 50)
print("MoE Transformer 验证")
print("=" * 50)

model = MoETransformer(config)

input_ids = torch.randint(0, config.vocab_size, (2, 32))  # batch=2, seq=32
model.train()
logits, aux = model(input_ids)

print(f"\n输入 shape: {input_ids.shape}")
print(f"Logits shape: {logits.shape}  (期望 [2, 32, {config.vocab_size}])")

print(f"\n模型结构 (MoE 层标记):")
for i, layer in enumerate(model.layers):
    layer_type = "MoE" if layer.use_moe else "Dense"
    print(f"  Layer {i}: {layer_type}")

if aux:
    print(f"\n辅助损失:")
    print(f"  平均 Load Balance Loss: {aux['load_balance'].item():.6f}")
    print(f"  平均 Z-Loss: {aux['z_loss'].item():.6f}")
    print(f"  各 MoE 层各专家 Top-1 计数:\n{aux['expert_counts'].int()}")

assert logits.shape == (2, 32, config.vocab_size)
print(f"\n✅ MoE Transformer 验证通过!")

## Part 6：验证 — Routing 分布可视化

可视化路由分布，观察：
1. 各专家被选中的频率是否均匀
2. 不同 token 的路由权重分布
3. 路由是否呈现"赢者通吃"模式

In [ ]:
def visualize_routing(model: MoETransformer, input_ids: torch.Tensor):
    """可视化 MoE 路由分布"""
    model.eval()
    
    # 收集所有 MoE 层的路由信息
    routing_data = []
    
    x = model.embedding(input_ids)
    for i, layer in enumerate(model.layers):
        h = x + layer.attention(layer.attn_norm(x))
        normed = layer.ffn_norm(h)
        
        if layer.use_moe:
            x_flat = normed.view(-1, model.config.d_model)
            with torch.no_grad():
                logits = layer.ffn.router.gate(x_flat)
                gates = F.softmax(logits, dim=-1)
                topk_w, topk_idx = torch.topk(gates, model.config.top_k, dim=-1)
                routing_data.append({
                    'layer': i,
                    'gates': gates.cpu(),
                    'topk_idx': topk_idx.cpu(),
                    'topk_weights': topk_w.cpu()
                })
            _, aux = layer.ffn(normed)
            x = h + _
        else:
            x = h + layer.ffn(normed)
    
    # 可视化
    n_moe_layers = len(routing_data)
    fig, axes = plt.subplots(n_moe_layers, 3, figsize=(15, 4 * n_moe_layers))
    if n_moe_layers == 1:
        axes = axes.reshape(1, -1)
    
    for li, rd in enumerate(routing_data):
        # 1. 各专家被选为 Top-1 的频率
        top1 = rd['topk_idx'][:, 0]
        counts = torch.bincount(top1, minlength=model.config.num_experts)
        axes[li, 0].bar(range(model.config.num_experts), counts.numpy())
        axes[li, 0].set_title(f'Layer {rd["layer"]}: Expert Selection (Top-1)')
        axes[li, 0].set_xlabel('Expert ID')
        axes[li, 0].set_ylabel('Token Count')
        ideal = len(top1) / model.config.num_experts
        axes[li, 0].axhline(y=ideal, color='r', linestyle='--', label=f'Ideal={ideal:.0f}')
        axes[li, 0].legend()
        
        # 2. 路由概率热力图 (前 32 个 token)
        gates_subset = rd['gates'][:min(32, rd['gates'].shape[0])]
        im = axes[li, 1].imshow(gates_subset.numpy(), aspect='auto', cmap='YlOrRd')
        axes[li, 1].set_title(f'Layer {rd["layer"]}: Routing Probabilities')
        axes[li, 1].set_xlabel('Expert ID')
        axes[li, 1].set_ylabel('Token ID')
        plt.colorbar(im, ax=axes[li, 1])
        
        # 3. Top-K 权重分布
        for k in range(model.config.top_k):
            weights_k = rd['topk_weights'][:, k].numpy()
            axes[li, 2].hist(weights_k, bins=30, alpha=0.6, label=f'Top-{k+1}')
        axes[li, 2].set_title(f'Layer {rd["layer"]}: Top-K Weight Distribution')
        axes[li, 2].set_xlabel('Weight Value')
        axes[li, 2].set_ylabel('Count')
        axes[li, 2].legend()
    
    plt.tight_layout()
    plt.savefig('moe_routing_visualization.png', dpi=100, bbox_inches='tight')
    plt.show()
    print("\n路由分布可视化已保存到 moe_routing_visualization.png")


# 运行可视化
test_input = torch.randint(0, config.vocab_size, (4, 64))  # batch=4, seq=64
visualize_routing(model, test_input)

## Part 7：验证 — 负载均衡 Loss 有效性

对比有/无辅助负载均衡 Loss 时的路由分布差异。

实验设计：
1. 训练两个小 MoE 模型（几步即可），一个有辅助 loss，一个没有
2. 比较训练后的专家利用率

In [ ]:
def train_moe_steps(config: MoEConfig, num_steps: int = 100, use_aux_loss: bool = True):
    """训练 MoE 模型若干步，返回专家利用率统计"""
    cfg = MoEConfig(
        d_model=64, n_heads=4, n_layers=2,
        num_experts=8, top_k=2, expert_ffn_dim=128,
        aux_loss_alpha=0.01 if use_aux_loss else 0.0,
        z_loss_weight=0.001 if use_aux_loss else 0.0,
        vocab_size=100, max_seq_len=32, moe_every_n_layers=1
    )
    
    model = MoETransformer(cfg)
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    
    expert_count_history = []
    loss_history = []
    
    for step in range(num_steps):
        input_ids = torch.randint(0, cfg.vocab_size, (8, 32))
        target = torch.randint(0, cfg.vocab_size, (8, 32))
        
        logits, aux = model(input_ids)
        
        task_loss = F.cross_entropy(logits.view(-1, cfg.vocab_size), target.view(-1))
        total_loss = task_loss
        
        if aux and use_aux_loss:
            total_loss = total_loss + aux['load_balance'] + aux['z_loss']
        
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
        
        if aux:
            counts = aux['expert_counts'].sum(dim=0)  # 合并所有 MoE 层
            expert_count_history.append(counts.detach().cpu())
        loss_history.append(task_loss.item())
    
    return expert_count_history, loss_history


# 对比实验
print("训练有辅助 loss 的 MoE 模型...")
counts_with, loss_with = train_moe_steps(config, num_steps=200, use_aux_loss=True)

print("训练无辅助 loss 的 MoE 模型...")
counts_without, loss_without = train_moe_steps(config, num_steps=200, use_aux_loss=False)

# 可视化对比
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. 最终专家分布对比
final_with = counts_with[-1].numpy() if counts_with else np.zeros(8)
final_without = counts_without[-1].numpy() if counts_without else np.zeros(8)

x_pos = np.arange(8)
width = 0.35
axes[0].bar(x_pos - width/2, final_with, width, label='With Aux Loss', color='steelblue')
axes[0].bar(x_pos + width/2, final_without, width, label='Without Aux Loss', color='coral')
axes[0].set_xlabel('Expert ID')
axes[0].set_ylabel('Token Count')
axes[0].set_title('Final Expert Distribution')
axes[0].legend()

# 2. 专家利用率随训练变化（标准差越小越均衡）
std_with = [c.std().item() for c in counts_with]
std_without = [c.std().item() for c in counts_without]
axes[1].plot(std_with, label='With Aux Loss', color='steelblue')
axes[1].plot(std_without, label='Without Aux Loss', color='coral')
axes[1].set_xlabel('Training Step')
axes[1].set_ylabel('Std of Expert Counts')
axes[1].set_title('Expert Balance (lower = more balanced)')
axes[1].legend()

# 3. 训练 Loss 曲线
axes[2].plot(loss_with, label='With Aux Loss', color='steelblue', alpha=0.7)
axes[2].plot(loss_without, label='Without Aux Loss', color='coral', alpha=0.7)
axes[2].set_xlabel('Training Step')
axes[2].set_ylabel('Task Loss')
axes[2].set_title('Training Loss')
axes[2].legend()

plt.tight_layout()
plt.savefig('moe_load_balance_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"\n有辅助 loss — 专家分布标准差 (最终): {std_with[-1]:.2f}")
print(f"无辅助 loss — 专家分布标准差 (最终): {std_without[-1]:.2f}")
print(f"\n{'✅ 辅助 loss 有效减少了专家负载不均衡!' if std_with[-1] < std_without[-1] else '⚠️ 差异不明显（可能需要更多训练步数）'}")

## Part 8：MoE 参数量分析

区分 MoE 的**总参数**和**激活参数**：

$$N_{\text{total}} = N_{\text{non-MoE}} + E \times N_{\text{expert}}$$

$$N_{\text{active}} = N_{\text{non-MoE}} + K \times N_{\text{expert}}$$

显存按 $N_{\text{total}}$ 计，计算量按 $N_{\text{active}}$ 计。

In [ ]:
def analyze_moe_params(model: MoETransformer, config: MoEConfig):
    """分析 MoE 模型的参数量：总参数 vs 激活参数"""
    
    total_params = 0
    non_moe_params = 0
    moe_expert_params = 0
    shared_expert_params = 0
    
    for name, param in model.named_parameters():
        total_params += param.numel()
        if '.experts.' in name:
            moe_expert_params += param.numel()
        elif '.shared_experts.' in name:
            shared_expert_params += param.numel()
        else:
            non_moe_params += param.numel()
    
    # 每个专家的参数量
    per_expert_params = moe_expert_params // config.num_experts if config.num_experts > 0 else 0
    
    # MoE 层的数量
    n_moe_layers = sum(1 for l in model.layers if l.use_moe)
    per_expert_per_layer = per_expert_params // n_moe_layers if n_moe_layers > 0 else 0
    
    # 激活参数 = 非 MoE 参数 + Top-K 个专家参数
    active_params = non_moe_params + shared_expert_params + (config.top_k * per_expert_params)
    
    print("=" * 60)
    print("MoE 参数量分析")
    print("=" * 60)
    print(f"\n模型结构:")
    print(f"  总层数: {config.n_layers}")
    print(f"  MoE 层数: {n_moe_layers}")
    print(f"  Dense 层数: {config.n_layers - n_moe_layers}")
    print(f"  专家数 E: {config.num_experts}")
    print(f"  Top-K: {config.top_k}")
    
    print(f"\n参数量分解:")
    print(f"  非 MoE 参数 (Attention + Embed + Norm + LM Head): {non_moe_params:>12,}")
    print(f"  路由专家参数 ({config.num_experts} 专家 × {n_moe_layers} 层):     {moe_expert_params:>12,}")
    print(f"  共享专家参数:                                     {shared_expert_params:>12,}")
    print(f"  每个专家参数 (每层):                               {per_expert_per_layer:>12,}")
    print(f"  {'─' * 52}")
    print(f"  总参数量 N_total:                                 {total_params:>12,}")
    print(f"  激活参数 N_active (Top-{config.top_k}):                       {active_params:>12,}")
    
    ratio = total_params / active_params if active_params > 0 else 0
    print(f"\n  N_total / N_active = {ratio:.2f}x")
    print(f"  显存需求按 {total_params:,} 参数计")
    print(f"  计算量按 {active_params:,} 参数计")
    
    # 对比等参数稠密模型
    print(f"\n对比等计算量稠密模型:")
    print(f"  MoE 总参数 {total_params:,} ≈ {total_params/1e6:.1f}M")
    print(f"  等效稠密模型参数 ≈ {active_params:,} ≈ {active_params/1e6:.1f}M")
    print(f"  MoE 用 {ratio:.1f}x 参数换取更强表示能力，但计算量相同")
    
    return {
        'total': total_params,
        'active': active_params,
        'non_moe': non_moe_params,
        'expert': moe_expert_params,
        'ratio': ratio
    }


param_info = analyze_moe_params(model, config)

### 模拟 Mixtral 8x7B 参数分析

用真实的 Mixtral 参数配置来验算参数量。

In [ ]:
def mixtral_param_analysis():
    """Mixtral 8x7B 参数量理论分析"""
    d = 4096         # 隐层维度
    d_ff = 14336     # FFN 中间维度
    n_heads = 32     # Q 头数
    n_kv_heads = 8   # KV 头数 (GQA)
    head_dim = d // n_heads  # 128
    L = 32           # 层数
    E = 8            # 专家数
    K = 2            # Top-K
    vocab = 32000    # 词表
    
    # 每层 Attention 参数
    attn_params = (
        d * n_heads * head_dim +       # W_Q
        d * n_kv_heads * head_dim +    # W_K
        d * n_kv_heads * head_dim +    # W_V
        n_heads * head_dim * d         # W_O
    )
    
    # 每个专家 FFN 参数 (SwiGLU: gate + up + down)
    per_expert_params = d * d_ff + d * d_ff + d_ff * d  # W_gate + W_up + W_down
    
    # 每层 MoE 参数 = Router + E 个专家
    router_params = d * E  # W_gate
    moe_layer_params = router_params + E * per_expert_params
    
    # 每层 Norm 参数
    norm_params = 2 * d  # 2 个 RMSNorm
    
    # Embedding + LM Head
    embed_params = vocab * d
    
    # 总参数
    total = L * (attn_params + moe_layer_params + norm_params) + 2 * embed_params
    
    # 激活参数（只用 K 个专家）
    active_per_layer = attn_params + router_params + K * per_expert_params + norm_params
    active = L * active_per_layer + 2 * embed_params
    
    print("=" * 60)
    print("Mixtral 8x7B 参数量分析")
    print("=" * 60)
    print(f"\n每层 Attention 参数: {attn_params / 1e6:.1f}M")
    print(f"每个专家 FFN 参数:   {per_expert_params / 1e6:.1f}M")
    print(f"每层 MoE 参数:       {moe_layer_params / 1e6:.1f}M  ({E} 专家)")
    print(f"Router 参数 (每层):  {router_params:,}")
    print(f"Embedding 参数:      {embed_params / 1e6:.1f}M")
    print(f"\n总参数: {total / 1e9:.1f}B")
    print(f"激活参数 (Top-{K}): {active / 1e9:.1f}B")
    print(f"比值: {total / active:.2f}x")
    print(f"\n显存 (FP16): {total * 2 / 1e9:.1f} GB")
    print(f"显存 (INT4): {total * 0.5 / 1e9:.1f} GB")

mixtral_param_analysis()

## Part 9：模拟 Expert Parallelism（AllToAll）

Expert Parallelism 将不同专家放在不同 GPU 上。核心通信是 **AllToAll**：
- 路由决定后，将 token 发送到对应专家所在的 GPU
- 专家计算后，将结果发送回原始 GPU

通信模式：$\text{AllToAll} \rightarrow \text{Expert Compute} \rightarrow \text{AllToAll}$

In [ ]:
def simulate_all_to_all(gpu_data: List[List[torch.Tensor]], 
                         routing: List[List[int]]) -> List[List[torch.Tensor]]:
    """
    模拟 AllToAll 通信：每个 GPU 将 token 发送到目标 GPU。
    
    Args:
        gpu_data: gpu_data[i][j] = GPU i 上要发给 GPU j 的 tokens
        routing: routing[i][j] = GPU i 的第 j 个 token 应去的 GPU
    Returns:
        recv_data: recv_data[i] = GPU i 收到的所有 tokens
    """
    n_gpus = len(gpu_data)
    recv_data = [[] for _ in range(n_gpus)]
    
    for src_gpu in range(n_gpus):
        for dst_gpu in range(n_gpus):
            if gpu_data[src_gpu][dst_gpu] is not None and len(gpu_data[src_gpu][dst_gpu]) > 0:
                recv_data[dst_gpu].append(gpu_data[src_gpu][dst_gpu])
    
    return recv_data


def simulate_expert_parallelism(tokens: torch.Tensor, router: TopKRouter,
                                  experts: nn.ModuleList, num_gpus: int = 4):
    """
    模拟 Expert Parallelism 的完整流程。
    
    Args:
        tokens: [T, D] — 所有 token
        router: 路由器
        experts: 所有专家
        num_gpus: GPU 数量
    """
    num_experts = len(experts)
    experts_per_gpu = num_experts // num_gpus
    T, D = tokens.shape
    tokens_per_gpu = T // num_gpus
    
    print(f"\n配置: {num_experts} 专家, {num_gpus} GPU, 每 GPU {experts_per_gpu} 专家")
    print(f"总 token 数: {T}, 每 GPU token 数: {tokens_per_gpu}")
    
    # Step 1: 每 GPU 持有一部分 token，做路由决策
    print("\n--- Step 1: 路由决策 ---")
    router.eval()
    with torch.no_grad():
        topk_idx, topk_w, _ = router(tokens)
    
    # 将 token 按 GPU 分组
    gpu_tokens = [tokens[i * tokens_per_gpu:(i + 1) * tokens_per_gpu] for i in range(num_gpus)]
    gpu_topk_idx = [topk_idx[i * tokens_per_gpu:(i + 1) * tokens_per_gpu] for i in range(num_gpus)]
    gpu_topk_w = [topk_w[i * tokens_per_gpu:(i + 1) * tokens_per_gpu] for i in range(num_gpus)]
    
    # Step 2: AllToAll — 将 token 发送到对应专家所在 GPU
    print("\n--- Step 2: AllToAll (Token → Expert GPU) ---")
    
    send_counts = torch.zeros(num_gpus, num_gpus, dtype=torch.long)
    
    for src_gpu in range(num_gpus):
        for token_i in range(tokens_per_gpu):
            for k in range(router.top_k):
                expert_id = gpu_topk_idx[src_gpu][token_i, k].item()
                dst_gpu = expert_id // experts_per_gpu
                send_counts[src_gpu, dst_gpu] += 1
    
    print("\nAllToAll 通信矩阵 (src_gpu → dst_gpu, token count):")
    header = "     " + "  ".join(f"GPU{j}" for j in range(num_gpus))
    print(header)
    for i in range(num_gpus):
        row = f"GPU{i}  " + "   ".join(f"{send_counts[i, j]:3d}" for j in range(num_gpus))
        print(row)
    
    # Step 3: Expert 计算
    print("\n--- Step 3: Expert 计算 ---")
    for gpu_id in range(num_gpus):
        received = send_counts[:, gpu_id].sum().item()
        expert_ids = list(range(gpu_id * experts_per_gpu, (gpu_id + 1) * experts_per_gpu))
        print(f"  GPU {gpu_id}: 收到 {received} 个 token, 使用 Expert {expert_ids}")
    
    # Step 4: AllToAll 回传结果
    print("\n--- Step 4: AllToAll (Expert GPU → Token GPU) ---")
    print("  将计算结果发送回原始 GPU（通信矩阵的转置）")
    
    # 通信量分析
    total_comm = send_counts.sum().item()
    local_comm = send_counts.diagonal().sum().item()
    cross_comm = total_comm - local_comm
    print(f"\n通信分析:")
    print(f"  总路由 token 数: {total_comm}")
    print(f"  本地计算 (无需通信): {local_comm} ({local_comm/total_comm:.1%})")
    print(f"  跨 GPU 通信: {cross_comm} ({cross_comm/total_comm:.1%})")
    print(f"  每个 token 通信量: {D * 2} bytes (FP16) × 2 (来回) = {D * 4} bytes")
    print(f"  总通信量: {cross_comm * D * 4 / 1024:.1f} KB")


# 运行 EP 模拟
print("=" * 60)
print("Expert Parallelism 模拟")
print("=" * 60)

test_tokens = torch.randn(64, config.d_model)  # 64 tokens
simulate_expert_parallelism(test_tokens, model.layers[1].ffn.router, 
                            model.layers[1].ffn.experts, num_gpus=4)

## Part 10：端到端 MoE 训练循环

实现完整的 MoE 训练循环，包括：
1. Language Modeling 任务损失
2. 辅助负载均衡损失
3. Z-Loss
4. 训练监控（loss 曲线 + 专家利用率）

$$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{LM}} + \mathcal{L}_{\text{aux}} + \mathcal{L}_z$$

In [ ]:
def create_synthetic_data(vocab_size: int, seq_len: int, batch_size: int, num_batches: int):
    """生成简单的模式化训练数据（方便验证模型能否学会）"""
    data = []
    for _ in range(num_batches):
        input_ids = torch.randint(0, vocab_size, (batch_size, seq_len))
        # 目标 = 输入右移一位（简单的 next-token prediction）
        targets = torch.roll(input_ids, -1, dims=1)
        targets[:, -1] = 0  # 最后一个位置设为 pad
        data.append((input_ids, targets))
    return data


def train_moe_model(config: MoEConfig, num_epochs: int = 3, num_batches: int = 50,
                     batch_size: int = 8, seq_len: int = 32, lr: float = 3e-4):
    """
    端到端 MoE 训练循环。
    
    Returns:
        model, metrics_history
    """
    model = MoETransformer(config)
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    
    train_data = create_synthetic_data(config.vocab_size, seq_len, batch_size, num_batches)
    
    metrics = {
        'total_loss': [], 'task_loss': [], 'aux_loss': [], 'z_loss': [],
        'expert_entropy': [], 'expert_counts_per_step': []
    }
    
    total_steps = 0
    for epoch in range(num_epochs):
        epoch_task_loss = 0
        for batch_idx, (input_ids, targets) in enumerate(train_data):
            logits, aux = model(input_ids)
            
            # 任务损失
            task_loss = F.cross_entropy(
                logits.view(-1, config.vocab_size),
                targets.view(-1)
            )
            
            # 总损失 = 任务 + 辅助
            total_loss = task_loss
            aux_loss_val = 0.0
            z_loss_val = 0.0
            
            if aux:
                total_loss = total_loss + aux['load_balance'] + aux['z_loss']
                aux_loss_val = aux['load_balance'].item()
                z_loss_val = aux['z_loss'].item()
            
            optimizer.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            # 记录指标
            metrics['total_loss'].append(total_loss.item())
            metrics['task_loss'].append(task_loss.item())
            metrics['aux_loss'].append(aux_loss_val)
            metrics['z_loss'].append(z_loss_val)
            
            if aux and 'expert_counts' in aux:
                counts = aux['expert_counts'].sum(dim=0).detach().cpu()
                metrics['expert_counts_per_step'].append(counts)
                probs = counts / counts.sum()
                entropy = -(probs * (probs + 1e-9).log()).sum().item()
                metrics['expert_entropy'].append(entropy)
            
            epoch_task_loss += task_loss.item()
            total_steps += 1
        
        avg_loss = epoch_task_loss / num_batches
        print(f"Epoch {epoch+1}/{num_epochs}  |  Task Loss: {avg_loss:.4f}  |  "
              f"Aux: {metrics['aux_loss'][-1]:.6f}  |  Z: {metrics['z_loss'][-1]:.6f}")
    
    return model, metrics


# 训练
print("=" * 60)
print("端到端 MoE 训练")
print("=" * 60)

train_config = MoEConfig(
    d_model=128, n_heads=4, n_layers=4,
    num_experts=8, top_k=2, expert_ffn_dim=256,
    aux_loss_alpha=0.01, z_loss_weight=0.001,
    vocab_size=500, max_seq_len=32,
    moe_every_n_layers=2
)

trained_model, metrics = train_moe_model(
    train_config, num_epochs=5, num_batches=50, batch_size=8, seq_len=32
)

In [ ]:
# 训练监控可视化
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. 总损失 + 任务损失
axes[0, 0].plot(metrics['total_loss'], label='Total Loss', alpha=0.7)
axes[0, 0].plot(metrics['task_loss'], label='Task Loss', alpha=0.7)
axes[0, 0].set_xlabel('Step')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Training Loss')
axes[0, 0].legend()

# 2. 辅助损失
axes[0, 1].plot(metrics['aux_loss'], label='Load Balance Loss', color='orange', alpha=0.7)
axes[0, 1].plot(metrics['z_loss'], label='Z-Loss', color='green', alpha=0.7)
axes[0, 1].set_xlabel('Step')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].set_title('Auxiliary Losses')
axes[0, 1].legend()

# 3. 专家利用率熵（越高越均衡）
if metrics['expert_entropy']:
    max_entropy = math.log(train_config.num_experts)
    axes[1, 0].plot(metrics['expert_entropy'], color='purple', alpha=0.7)
    axes[1, 0].axhline(y=max_entropy, color='r', linestyle='--', 
                       label=f'Max Entropy (uniform) = {max_entropy:.2f}')
    axes[1, 0].set_xlabel('Step')
    axes[1, 0].set_ylabel('Entropy')
    axes[1, 0].set_title('Expert Selection Entropy (higher = more balanced)')
    axes[1, 0].legend()

# 4. 最终专家分布
if metrics['expert_counts_per_step']:
    # 最后 10 步的平均
    last_counts = torch.stack(metrics['expert_counts_per_step'][-10:]).mean(dim=0).numpy()
    axes[1, 1].bar(range(train_config.num_experts), last_counts, color='steelblue')
    ideal = last_counts.sum() / train_config.num_experts
    axes[1, 1].axhline(y=ideal, color='r', linestyle='--', label=f'Ideal = {ideal:.0f}')
    axes[1, 1].set_xlabel('Expert ID')
    axes[1, 1].set_ylabel('Token Count (avg last 10 steps)')
    axes[1, 1].set_title('Final Expert Distribution')
    axes[1, 1].legend()

plt.tight_layout()
plt.savefig('moe_training_metrics.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n训练监控可视化已保存到 moe_training_metrics.png")
print(f"\n最终指标:")
print(f"  Task Loss (最后 10 步均值): {np.mean(metrics['task_loss'][-10:]):.4f}")
print(f"  Expert Entropy (最后 10 步均值): {np.mean(metrics['expert_entropy'][-10:]):.4f}")
print(f"  Max Entropy (完全均衡): {math.log(train_config.num_experts):.4f}")

## 总结

### 本 Notebook 实现的核心组件

| Part | 组件 | 关键实现 |
|------|------|--------|
| 2 | TopKRouter | Softmax → TopK → 归一化 + Noisy Gating + 辅助 Loss |
| 3 | Expert | GELU FFN + SwiGLU FFN |
| 4 | MoE Layer | Router + 多专家分发 + 加权求和 + 共享专家 |
| 5 | MoE Transformer | Attention + MoE Block (交替放置) + 辅助 Loss 聚合 |
| 6 | Routing 可视化 | 专家频率 + 概率热力图 + 权重分布 |
| 7 | 负载均衡验证 | 有/无辅助 Loss 对比实验 |
| 8 | 参数分析 | 总参数 vs 激活参数 + Mixtral 参数计算 |
| 9 | EP 模拟 | AllToAll 通信矩阵 + 通信量分析 |
| 10 | 训练循环 | LM Loss + Aux Loss + 训练监控 |

### 与 Day 5 理论的对应

- **Top-K Routing** → Day 5 第三节路由机制
- **辅助 Loss** → Day 5 第四节负载均衡
- **SwiGLU Expert** → Day 5 第五节 Mixtral
- **共享专家** → Day 5 第六节 DeepSeek-V2
- **EP 模拟** → Day 4 第八节 Expert Parallelism

### 下一步

Day 7 将精读 Megatron-LM 和 ZeRO 论文，复盘本周全部内容（DP → ZeRO → TP → PP → CP → EP → MoE），建立完整的分布式训练知识体系。